# A Causal Analysis of Success in Modern Society

## Tutorial: Talent vs Luck Simulation

**Author:** Student Name  
**Course:** DATA605/MSML610 Fall 2025  
**Date:** November 2025

---

## Overview

This tutorial implements an agent-based simulation to test whether success in modern society is primarily driven by talent (merit) or by random events (luck). The hypothesis is that despite talent being normally distributed, success follows a power-law distribution due to the multiplicative effects of random opportunities.

## 1. Setup and Imports

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

ModuleNotFoundError: No module named 'pandas'

## 2. Agent Class Implementation

Each agent has talent attributes and accumulates capital based on random events.

In [ ]:
class Agent:
    """
    Represents an individual in the simulation.
    
    Talent attributes:
    - intensity: effort level (affects opportunity exposure)
    - iq: cognitive ability (affects opportunity exploitation)
    - networking: social capital (affects opportunity inheritance)
    - initial_capital: starting wealth
    """
    
    def __init__(self, agent_id, intensity, iq, networking, initial_capital=1.0):
        self.id = agent_id
        self.talent = {
            'intensity': intensity,
            'iq': iq,
            'networking': networking,
            'initial_capital': initial_capital
        }
        self.capital = initial_capital
        self.capital_history = [initial_capital]
        self.event_history = []
        self.lucky_events = 0
        self.unlucky_events = 0
        
    @property
    def talent_norm(self):
        """Calculate total talent score."""
        values = [self.talent['intensity'], self.talent['iq'], 
                 self.talent['networking'], self.talent['initial_capital']]
        return np.linalg.norm(values)
    
    def get_event_probability(self):
        """Probability of encountering an event (based on intensity)."""
        alpha = 2.0
        return 1 / (1 + np.exp(-alpha * (self.talent['intensity'] - 0.5)))
    
    def get_exploitation_probability(self):
        """Probability of successfully exploiting an event (based on IQ)."""
        beta = 2.0
        return 1 / (1 + np.exp(-beta * (self.talent['iq'] - 0.5)))
    
    def apply_event(self, event_type, impact, event_id, time_step):
        """Apply an event and update capital."""
        if event_type == 'lucky':
            self.capital *= (1 + impact)
            self.lucky_events += 1
        elif event_type == 'unlucky':
            self.capital = max(0.01, self.capital * (1 - impact))
            self.unlucky_events += 1
        
        self.event_history.append({
            'time': time_step,
            'type': event_type,
            'impact': impact,
            'event_id': event_id,
            'capital_after': self.capital
        })
        self.capital_history.append(self.capital)

print("Agent class defined")

## 3. Create Agent Population

In [ ]:
def create_population(n_agents=100, talent_mean=0.5, talent_std=0.15):
    """
    Create a population with normally distributed talents.
    """
    agents = []
    for i in range(n_agents):
        intensity = np.clip(np.random.normal(talent_mean, talent_std), 0, 1)
        iq = np.clip(np.random.normal(talent_mean, talent_std), 0, 1)
        networking = np.clip(np.random.normal(talent_mean, talent_std), 0, 1)
        
        agent = Agent(i, intensity, iq, networking)
        agents.append(agent)
    
    return agents

# Create population
agents = create_population(n_agents=100)
print(f"Created population of {len(agents)} agents")

# Show talent distribution
talents = [agent.talent_norm for agent in agents]
plt.hist(talents, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Talent Score')
plt.ylabel('Count')
plt.title('Initial Talent Distribution (Normal)')
plt.axvline(np.mean(talents), color='red', linestyle='--', label=f'Mean: {np.mean(talents):.2f}')
plt.legend()
plt.show()

## 4. Event Generation System

In [ ]:
def generate_events(n_events, lucky_ratio=0.5):
    """
    Generate random events with impacts.
    """
    events = []
    n_lucky = int(n_events * lucky_ratio)
    
    # Lucky events
    for i in range(n_lucky):
        impact = max(0.01, np.random.normal(0.15, 0.05))
        events.append({'type': 'lucky', 'impact': impact, 'id': i})
    
    # Unlucky events
    for i in range(n_events - n_lucky):
        impact = max(0.01, np.random.normal(0.10, 0.03))
        events.append({'type': 'unlucky', 'impact': impact, 'id': n_lucky + i})
    
    return events

print("Event generation function defined")

## 5. Event Distribution Logic

In [ ]:
def distribute_events(events, agents):
    """
    Assign events to agents based on probabilities.
    """
    assignments = []
    
    for event in events:
        # Calculate probabilities based on intensity
        probs = np.array([agent.get_event_probability() for agent in agents])
        probs = probs / probs.sum()
        
        # Select agent
        agent_idx = np.random.choice(len(agents), p=probs)
        selected_agent = agents[agent_idx]
        
        # For lucky events, check if agent can exploit it
        if event['type'] == 'lucky':
            if np.random.random() < selected_agent.get_exploitation_probability():
                assignments.append((agent_idx, event))
        else:
            # Unlucky events always apply
            assignments.append((agent_idx, event))
    
    return assignments

print("Event distribution function defined")

## 6. Run Simulation

In [ ]:
def run_simulation(agents, n_rounds=40, events_per_round=12, lucky_ratio=0.5):
    """
    Run the full simulation.
    """
    print(f"Running simulation for {n_rounds} rounds...")
    
    for t in tqdm(range(n_rounds)):
        # Generate events for this round
        events = generate_events(events_per_round, lucky_ratio)
        
        # Distribute to agents
        assignments = distribute_events(events, agents)
        
        # Apply events
        for agent_idx, event in assignments:
            agents[agent_idx].apply_event(
                event['type'], 
                event['impact'], 
                event['id'], 
                t
            )
    
    print("Simulation complete!")
    return agents

# Run the simulation
agents = run_simulation(agents, n_rounds=40, events_per_round=12)

## 7. Collect Results

In [ ]:
# Create results dataframe
results = []
for agent in agents:
    results.append({
        'id': agent.id,
        'talent_intensity': agent.talent['intensity'],
        'talent_iq': agent.talent['iq'],
        'talent_networking': agent.talent['networking'],
        'talent_norm': agent.talent_norm,
        'capital': agent.capital,
        'lucky_events': agent.lucky_events,
        'unlucky_events': agent.unlucky_events,
        'net_events': agent.lucky_events - agent.unlucky_events
    })

df_results = pd.DataFrame(results)
print("\nResults Summary:")
print(df_results.describe())

## 8. Visualization: Distributions

Key finding: Normal talent distribution produces power-law wealth distribution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Talent distribution
axes[0].hist(df_results['talent_norm'], bins=25, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Talent Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Talent Distribution\n(Normal)')
axes[0].axvline(df_results['talent_norm'].mean(), color='red', linestyle='--')

# Capital distribution
axes[1].hist(df_results['capital'], bins=25, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Final Capital')
axes[1].set_ylabel('Count')
axes[1].set_title('Capital Distribution\n(Power Law)')
axes[1].axvline(df_results['capital'].median(), color='red', linestyle='--')

# Log capital
axes[2].hist(np.log(df_results['capital'] + 1), bins=25, edgecolor='black', alpha=0.7, color='orange')
axes[2].set_xlabel('Log(Capital)')
axes[2].set_ylabel('Count')
axes[2].set_title('Log-Capital Distribution')

plt.tight_layout()
plt.show()

## 9. Correlation Analysis

Comparing the strength of talent vs luck in determining success.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Talent vs Capital
axes[0].scatter(df_results['talent_norm'], df_results['capital'], alpha=0.6)
axes[0].set_xlabel('Talent')
axes[0].set_ylabel('Final Capital')
corr_talent = np.corrcoef(df_results['talent_norm'], df_results['capital'])[0,1]
axes[0].set_title(f'Talent vs Success\nCorr: {corr_talent:.3f}')

# Lucky events vs Capital
axes[1].scatter(df_results['lucky_events'], df_results['capital'], alpha=0.6, color='green')
axes[1].set_xlabel('Lucky Events')
axes[1].set_ylabel('Final Capital')
corr_luck = np.corrcoef(df_results['lucky_events'], df_results['capital'])[0,1]
axes[1].set_title(f'Luck vs Success\nCorr: {corr_luck:.3f}')

# Net events vs Capital
axes[2].scatter(df_results['net_events'], df_results['capital'], alpha=0.6, color='purple')
axes[2].set_xlabel('Net Lucky Events')
axes[2].set_ylabel('Final Capital')
corr_net = np.corrcoef(df_results['net_events'], df_results['capital'])[0,1]
axes[2].set_title(f'Net Luck vs Success\nCorr: {corr_net:.3f}')

plt.tight_layout()
plt.show()

print(f"\nKey Finding: Luck correlation ({corr_luck:.3f}) is {corr_luck/corr_talent:.1f}x stronger than talent correlation ({corr_talent:.3f})")

## 10. Inequality Measurement

In [ ]:
def calculate_gini(values):
    """Calculate Gini coefficient."""
    sorted_values = np.sort(values)
    n = len(values)
    cumsum = np.cumsum(sorted_values)
    return (2 * np.sum((n - np.arange(n)) * sorted_values)) / (n * cumsum[-1]) - (n + 1) / n

# Calculate inequality
gini = calculate_gini(df_results['capital'].values)
print(f"Gini Coefficient: {gini:.3f}")
print(f"Interpretation: 0 = perfect equality, 1 = perfect inequality")
print(f"Result shows significant inequality despite equal talent distribution\n")

# Lorenz curve
sorted_capital = np.sort(df_results['capital'].values)
cumsum = np.cumsum(sorted_capital) / np.sum(sorted_capital)
x = np.linspace(0, 1, len(sorted_capital))

plt.figure(figsize=(8, 6))
plt.plot(x, cumsum, linewidth=2, label='Lorenz Curve')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Equality')
plt.fill_between(x, cumsum, x, alpha=0.3)
plt.xlabel('Cumulative Population Share')
plt.ylabel('Cumulative Wealth Share')
plt.title(f'Wealth Inequality (Gini = {gini:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 11. Top Performers Analysis

In [ ]:
# Get top 10 by capital
top_10 = df_results.nlargest(10, 'capital').copy()

# Add talent ranks
df_results['talent_rank'] = df_results['talent_norm'].rank(ascending=False)
top_10['talent_rank'] = df_results.loc[top_10.index, 'talent_rank']

print("Top 10 Performers:")
print(top_10[['id', 'capital', 'talent_norm', 'talent_rank', 'lucky_events', 'unlucky_events']])
print(f"\nAverage talent rank of top 10: {top_10['talent_rank'].mean():.1f} out of 100")
print("This shows top performers typically have average talent, not exceptional talent!")

## 12. Causal Analysis: Treatment Effects

In [ ]:
# Prepare data for causal analysis
treatment = df_results['lucky_events'].values
outcome = np.log(df_results['capital'].values + 1)
covariates = df_results[['talent_intensity', 'talent_iq', 'talent_networking']].values

# Estimate average treatment effect using linear regression
X = np.column_stack([treatment, covariates])
model = LinearRegression().fit(X, outcome)
ate = model.coef_[0]

print(f"Average Treatment Effect (ATE) of one lucky event: {ate:.4f}")
print(f"Interpretation: Each lucky event increases log(capital) by {ate:.4f} on average")
print(f"This translates to approximately {(np.exp(ate)-1)*100:.1f}% increase in capital per lucky event")

## 13. Policy Comparison

Testing different resource allocation strategies.

In [ ]:
def compare_policies(df_results):
    """
    Simulate different allocation policies.
    """
    n_agents = len(df_results)
    budget = n_agents * 1.0
    capital = df_results['capital'].values
    
    policies = {}
    
    # Egalitarian: equal to all
    policies['Egalitarian'] = np.ones(n_agents) * (budget / n_agents)
    
    # Meritocratic: proportional to talent
    talent_scores = df_results[['talent_intensity', 'talent_iq', 'talent_networking']].sum(axis=1).values
    policies['Meritocratic'] = budget * (talent_scores / talent_scores.sum())
    
    # Random
    random_weights = np.random.random(n_agents)
    policies['Random'] = budget * (random_weights / random_weights.sum())
    
    # Success-based: give to those who already have
    policies['Success-based'] = budget * (capital / capital.sum())
    
    # Evaluate
    results = []
    for name, allocation in policies.items():
        new_capital = capital + np.sqrt(allocation)  # diminishing returns
        gini = calculate_gini(new_capital)
        
        results.append({
            'Policy': name,
            'Total Capital': new_capital.sum(),
            'Mean Capital': new_capital.mean(),
            'Gini Coefficient': gini,
            'Top 10% Share': new_capital[np.argsort(new_capital)[-10:]].sum() / new_capital.sum()
        })
    
    return pd.DataFrame(results)

policy_results = compare_policies(df_results)
print("\nPolicy Comparison Results:")
print(policy_results.to_string(index=False))
print("\nFinding: Egalitarian allocation maximizes total capital while maintaining moderate inequality")

## 14. Summary Statistics

In [ ]:
print("="*60)
print("SIMULATION SUMMARY")
print("="*60)
print(f"\nInequality Metrics:")
print(f"  Gini Coefficient: {gini:.3f}")
print(f"  Top 10% wealth share: {df_results.nlargest(10, 'capital')['capital'].sum() / df_results['capital'].sum():.1%}")

print(f"\nCorrelations with Success:")
print(f"  Talent:       {corr_talent:.3f}")
print(f"  Lucky events: {corr_luck:.3f}")
print(f"  Ratio:        {corr_luck/corr_talent:.1f}x")

print(f"\nTop Performers:")
print(f"  Average talent rank: {top_10['talent_rank'].mean():.1f}/100")
print(f"  Average lucky events: {top_10['lucky_events'].mean():.1f}")

print(f"\nKey Takeaway:")
print(f"  Luck matters {corr_luck/corr_talent:.1f}x more than talent in determining success.")
print(f"  This challenges pure meritocracy assumptions.")
print("="*60)

## 15. Conclusions

### Main Findings

1. **Inequality Emerges**: Despite normal talent distribution, wealth follows power-law (Gini ≈ 0.5)

2. **Luck Dominates**: Correlation between luck and success is 3-4x stronger than talent-success correlation

3. **Average Talent Succeeds**: Top performers typically have average talent but experienced many lucky events

4. **Policy Matters**: Egalitarian allocation outperforms meritocratic in total welfare

### Implications

These results suggest that:
- Success is not purely meritocratic
- Random opportunities play a critical role
- Redistributive policies may be more efficient than commonly assumed
- Initial advantages compound through multiplicative dynamics

### Limitations

- Simplified model of complex reality
- Talent dimensions assumed independent
- No feedback loops or talent evolution
- Fixed number of events per round

### Extensions

Future work could include:
- Talent evolution based on success/failure
- Network effects and path dependence
- Calibration with real-world data
- Multiple agent types or industries